# 01 · Calibration: reproducing CSEP's FY2020-21 gap decomposition

The base year must reproduce the cash-basis gap of ₹1,04,091 crore and its five components reported by CSEP (source `C1`). If this notebook fails, the engine has drifted from the source.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from discom_sim import *
base = base_from_table(read_csv('baseline_all_india_fy2021.csv'))

In [2]:
row = simulate(base, Levers(end_year=2021)).loc[2021]
check = pd.DataFrame({
    'model (Rs crore)': [row[k] for k in ['D','C','S','RI','residual','cash_gap']],
    'CSEP C1 (Rs crore)': [base[f'check_{k}'] for k in ['D','C','S','RI','residual','cash_gap']],
}, index=['D','C','S','RI','residual','cash_gap']).round(0)
check['difference'] = check.iloc[:,0] - check.iloc[:,1]
check

,model (Rs crore),CSEP C1 (Rs crore),difference
D,18022.0,18022.0,0.0
C,22295.0,22295.0,0.0
S,20228.0,20228.0,0.0
RI,7236.0,7236.0,0.0
residual,36310.0,36310.0,0.0
cash_gap,104091.0,104091.0,0.0


In [3]:
assert check['difference'].abs().max() <= 15, 'Base year no longer matches CSEP'
print(f"ACS {row.acs:.2f} Rs/kWh sold, cash gap {row.gap_per_kwh:.2f} Rs/kWh sold (CSEP: 7.91 and 1.14)")

ACS 7.91 Rs/kWh sold, cash gap 1.14 Rs/kWh sold (CSEP: 7.91 and 1.14)


### AT&C on the model's cash basis
The model's AT&C for FY21 is about 22.2%, while CSEP reports 23.02% achieved. The difference comes from the collection-efficiency definition: the model counts subsidy received as collected revenue, with booked tariff plus booked subsidy as the denominator.

In [4]:
print(f"Model AT&C FY21: {row.atc_loss:.2%}")

Model AT&C FY21: 22.21%


### Historical reference points (C1)
The engine starts in FY21. These earlier CSEP values are kept for context and future back-casting.

In [5]:
read_csv('calibration_series_csep.csv')

,fy,acs_rs_per_kwh_sold,gap_incl_grants_rs_crore,gap_rs_per_kwh_sold,source_id
0,2007,3.93,21184,0.54,C1
1,2016,7.02,93137,1.27,C1
2,2020,7.69,83829,0.90,C1
3,2021,7.91,104091,1.14,C1
